# NVCL_KIT Standard Export of Scalars Demonstration

This notebook demonstrates how to extract VNIR raw count, SWIR normalised and TIR normalised mineral groups and write them to CSV files in a standardised format, similar to the Summary Screen in The Spectral Geologist (TSG).

Any boreholes which are missing data (e.g. boreholes without TIR) or which don't match the expected classifications (e.g. older datasets) are recorded in a log (scalar_failures_log.csv).

In [ ]:
# Import the necessary modules
from nvcl_kit.generators import gen_summary_dataframe
from nvcl_kit.param_builder import param_builder
from nvcl_kit.reader import NVCLReader
from nvcl_kit.xml_helpers import clean_xml_parse

# Initialise NVCL reader
# Note: change this to WA, SA, NT, QLD, TAS to connect to other state services
param = param_builder("NSW")
if not param:
    print("Error: failed to setup connection parameters")

reader = NVCLReader(param)
if not reader.wfs:
    print("Error: Cannot contact service")

# Define the output format per spectral range
DESC_OUTPUT_COLUMNS = ["BoreholeURI", "BoreholeID", "DatasetName", "StartDepth", "EndDepth", "ScalarSet", "ScalarLevel", "Algorithm", "InstrumentName"]
VNIR_OUTPUT_COLUMNS = ["Error", "SNR", "NULL", "MISC-SILICATE", "CARBONATE", "SULPHATE", "OXIDE", "SULPHIDE", "CLAY-CU", "NOTAROK", "INVALID"]
SWIR_OUTPUT_COLUMNS = ["Error", "SNR", "NULL",	"SILICA",	"KAOLIN",	"WHITE-MICA",	"SMECTITE",	"OTHER-ALOH",	"CHLORITE",	"DARK-MICA",	"AMPHIBOLE",	"SERPENTINE",	"OTHER-MGOH",	"EPIDOTE",	"TOURMALINE",	"CARBONATE",	"SULPHATE",	"PAL-SEP",	"NOTAROK",	"INVALID"]
TIR_OUTPUT_COLUMNS =  ["Error", "SNR", "NULL", "MISC-SILICATE", "SILICA", "K-FELDSPAR", "PLAGIOCLASE", "GARNET", "PYROXENE", "OLIVINE", "ZEOLITE", "KAOLIN", "WHITE-MICA", "SMECTITE", "OTHER-ALOH", "CHLORITE", "DARK-MICA", "AMPHIBOLE", "SERPENTINE", "OTHER-MGOH", "EPIDOTE", "TOURMALINE", "CARBONATE", "SULPHATE", "PHOSPHATE", "BORATE", "OXIDE", "PAL-SEP", "INVALID"]

def format_export_dataframe(df, bh_meta, dataset_meta, output_desc_cols, output_scalar_cols, spectral_range):
    """"
    Accepts a dataframe with mineral classification data and metadata, and formats it for export by:
    - Adding metadata columns (BoreholeURI, InstrumentName, DatasetName, ScalarSet, ScalarLevel, Algorithm)
    - Renaming SNR and Error columns to match the specified output format
    - Reordering columns to match the specified output format and adding any missing columns with NULL values
    """
    # If bh_meta.href is a string, convert it to https:// and then add to dataframe, otherwise set to None
    if isinstance(bh_meta.href, str):
        df["BoreholeURI"] = bh_meta.href.replace("http://", "https://")
    else:
        df["BoreholeURI"] = None
    df["InstrumentName"] = bh_meta.instrument_name
    df["DatasetName"] = bh_meta.dataset_name
    scalar_set = dataset_meta.get("scalar_set")
    df["ScalarSet"] = scalar_set
    df["ScalarLevel"] = dataset_meta.get("scalar_level", "Unknown")
    if "scalar_algorithm_version" in dataset_meta:
        df["Algorithm"] = f"{spectral_range} v{dataset_meta.get('scalar_algorithm_version')}"
    # rename error and SNR columns to exclude scalar set
    error_col = f"Error_{scalar_set}"
    snr_col = f"SNR_{scalar_set}"
    df.rename(columns={error_col: "Error", snr_col: "SNR"}, inplace=True)

    # Reorder columns to match TIR_OUTPUT_COLUMNS and add any missing columns with NULL values
    for col in output_scalar_cols:
        if col not in df.columns:
            df[col] = 0.0
    for col in output_desc_cols:
        if col not in df.columns:
            df[col] = None
    df = df[output_desc_cols + output_scalar_cols]


    return df

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime
import csv

BIN_SIZE = 1.0

# Setup export directory
export_dir = Path("standard-export")
if not os.path.exists(export_dir):
    os.makedirs(export_dir)

# Set up logging for failed scalar writes
log_file = export_dir / "scalar_failures_log.csv"
if not log_file.exists():
    with open(log_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Timestamp", "BoreholeID", "Message"])

# Get every borehole ID for the service
bh_meta = reader.get_feature_list()
print(f"Found {len(bh_meta)} boreholes for {reader.param_obj.PROV.upper()}")

for bh in bh_meta:
    id = bh.nvcl_id
    print(f"Processing Borehole: {id}")

    # Make an output directory for the borehole
    if not os.path.exists(export_dir / id):
        os.makedirs(export_dir / id)
    elif len(os.listdir(export_dir / id)) > 0:
        print(f"Output already exists for {id}, skipping...")
        continue

    # Get additional metadata for the borehole from the dataset endpoint
    bh_meta2 = reader.get_dataset_list(nvcl_id=id)[-1]
    if hasattr(bh_meta2, "dataset_name"):
        setattr(bh, "dataset_name", bh_meta2.dataset_name)
    if hasattr(bh_meta2, "description"):
        root = clean_xml_parse(bh_meta2.description)
        setattr(bh, "instrument_name", root.findtext("./InstrumentName", default="Unknown"))
    else:
        setattr(bh, "instrument_name", "Unknown")

    # EXPORT VNIR RAW COUNT
    # ==================
    for scalar_set in ["dTSAV", "uTSAV", "sTSAV"]:
        OUTPUT_FILE = export_dir / id / f"{id}_{scalar_set}_group_1.0m.csv"
        if not os.path.exists(OUTPUT_FILE):
            try:
                resp = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[id], scalar_set=scalar_set, scalar_level="group", start_depth="floor", weighted=False, include_srss=True, include_snr=True, percent=False, resolution=BIN_SIZE, continue_on_missing=True), None)
            except Exception as e:
                message = f"Error occurred while generating {id}_{scalar_set}_group_1.0m.csv: {e}"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "VNIR: " + message])
                continue

            if resp is not None:
                meta, df = resp

                # If meta["classifications"].keys() contains values not in VNIR_OUTPUT_COLUMNS, write an error to the log file and continue
                invalid_classifications = [c for c in meta.get("classifications", {}).keys() if c not in VNIR_OUTPUT_COLUMNS]
                if invalid_classifications:
                    message = f"Incompatible classifications in metadata for {id}_{scalar_set}_group_1.0m.csv: {', '.join(invalid_classifications)}"
                    print(message)
                    with open(log_file, mode="a", newline="") as f:
                        writer = csv.writer(f)
                        writer.writerow([datetime.now().isoformat(), id, "VNIR: " + message])
                    break
                # Set up dataframe for export (e.g. rename columns, add metadata fields as columns, etc.)
                df = format_export_dataframe(df, bh, meta, DESC_OUTPUT_COLUMNS, VNIR_OUTPUT_COLUMNS, "VNIR")

                # Write the summary data (csv)
                df.to_csv(OUTPUT_FILE, index=False)
                # Write metadata (json)
                with open(export_dir / id / f"{id}_{scalar_set}_group_1.0m.json", "w") as f:
                    json.dump(meta, f, indent=2)
                break # If one of the VNIR scalar sets is successfully generated, skip the rest
            elif scalar_set == "sTSAV":
                message = f"Failed to generate: {id}_{scalar_set}_group_1.0m.csv"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "VNIR: " + message])

    # EXPORT SWIR normalised
    # ===================
    for scalar_set in ["dTSAS", "uTSAS", "sTSAS"]:
        OUTPUT_FILE = export_dir / id / f"{id}_{scalar_set}_group_1.0m.csv"
        if not os.path.exists(OUTPUT_FILE):
            try:
                resp = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[id], scalar_set=scalar_set, scalar_level="group", start_depth="floor", weighted=True, include_srss=True, include_snr=True, resolution=BIN_SIZE, continue_on_missing=True), None)
            except Exception as e:
                message = f"Error occurred while generating {id}_{scalar_set}_group_1.0m.csv: {e}"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "SWIR: " + message])
                continue

            if resp is not None:
                meta, df = resp
                # If meta["classifications"].keys() contains values not in SWIR_OUTPUT_COLUMNS, write an error to the log file and continue
                invalid_classifications = [c for c in meta.get("classifications", {}).keys() if c not in SWIR_OUTPUT_COLUMNS]
                if invalid_classifications:
                    message = f"Incompatible classifications in metadata for {id}_{scalar_set}_group_1.0m.csv: {', '.join(invalid_classifications)}"
                    print(message)
                    with open(log_file, mode="a", newline="") as f:
                        writer = csv.writer(f)
                        writer.writerow([datetime.now().isoformat(), id, "SWIR: " + message])
                    break
                # Set up dataframe for export (e.g. rename columns, add metadata fields as columns, etc.)
                df = format_export_dataframe(df, bh, meta, DESC_OUTPUT_COLUMNS, SWIR_OUTPUT_COLUMNS, "SWIR")
                # Write the summary data (csv)
                df.to_csv(OUTPUT_FILE, index=False)
                # Write metadata (json)
                with open(export_dir / id / f"{id}_{scalar_set}_group_1.0m.json", "w") as f:
                    json.dump(meta, f, indent=2)
                break # If one of the SWIR scalar sets is successfully generated, skip the rest
            elif scalar_set == "sTSAS":
                message = f"Failed to generate: {id}_{scalar_set}_group_1.0m.csv"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "SWIR: " + message])

    # EXPORT TIR normalised
    # ==================
    for scalar_set in ["djCLST", "ujCLST", "sjCLST", "dTSAT", "uTSAT", "sTSAT"]:
        OUTPUT_FILE = export_dir / id / f"{id}_{scalar_set}_group_1.0m.csv"
        if not os.path.exists(OUTPUT_FILE):
            try:
                resp = next(gen_summary_dataframe(reader=reader, nvcl_id_list=[id], scalar_set=scalar_set, scalar_level="group", start_depth="floor", weighted=True, include_srss=True, include_snr=True, resolution=BIN_SIZE, continue_on_missing=True), None)
            except Exception as e:
                message = f"Error occurred while generating {id}_{scalar_set}_group_1.0m.csv: {e}"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "TIR: " + message])
                continue

            if resp is not None:
                meta, df = resp
                # If meta["classifications"].keys() contains values not in TIR_OUTPUT_COLUMNS, write an error to the log file and continue
                invalid_classifications = [c for c in meta.get("classifications", {}).keys() if c not in TIR_OUTPUT_COLUMNS]
                if invalid_classifications:
                    message = f"Incompatible classifications in metadata for {id}_{scalar_set}_group_1.0m.csv: {', '.join(invalid_classifications)}"
                    print(message)
                    with open(log_file, mode="a", newline="") as f:
                        writer = csv.writer(f)
                        writer.writerow([datetime.now().isoformat(), id, "TIR: " + message])
                    break
                # Set up dataframe for export (e.g. rename columns, add metadata fields as columns, etc.)
                df = format_export_dataframe(df, bh, meta, DESC_OUTPUT_COLUMNS, TIR_OUTPUT_COLUMNS, "TIR")
                # Write the summary data (csv)
                df.to_csv(OUTPUT_FILE, index=False)
                # Write metadata (json)
                with open(export_dir / id / f"{id}_{scalar_set}_group_1.0m.json", "w") as f:
                    json.dump(meta, f, indent=2)
                break # If one of the TIR scalar sets is successfully generated, skip the rest
            elif scalar_set == "sTSAT":
                message = f"Failed to generate: {id}_{scalar_set}_group_1.0m.csv"
                print(message)
                with open(log_file, mode="a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([datetime.now().isoformat(), id, "TIR: " + message])